<div align="center">

# Laboratorio 9 · Embeddings

**Curso:** Natural Language Processing  
**Estudiante:** Jose Tánchez  
**Carné:** 20230005



# 1. Preparación de los corpus

Prepararemos las noticias y *Don Quijote de la Mancha* como **listas de oraciones**, cada una representada por una lista de tokens en su orden original. Las ventanas de contexto respetarán los límites de cada oración.

Ejecutar las celdas en orden. Las tablas y ejemplos quedarán en las salidas; el análisis de los resultados se redactará después de revisarlas.

## 1.1 Configuración y archivos

Se reutilizan `Lab1/df_total.csv` y `Lab5/don-quijote.txt`. El notebook puede ejecutarse desde la raíz del repositorio o desde `Lab9`.

Se requieren **pandas** y **NLTK**. La comprobación de SUC requiere además **gensim** y el archivo `embeddings-m-model.vec`. Si faltan paquetes, se pueden instalar desde una celda con `%pip install pandas nltk gensim`. El modelo preentrenado debe descargarse desde el enlace indicado en la guía y no debe subirse al repositorio.

In [1]:
from pathlib import Path
from collections import Counter
import os
import re
import unicodedata

import nltk
import pandas as pd
from IPython.display import display

candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ = next(
    (ruta for ruta in candidatas
     if (ruta / "Lab1/df_total.csv").is_file()
     and (ruta / "Lab5/don-quijote.txt").is_file()),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("Abre el notebook desde el repositorio NLP o desde Lab9.")

RUTA_NOTICIAS = RAIZ / "Lab1/df_total.csv"
RUTA_QUIJOTE = RAIZ / "Lab5/don-quijote.txt"
# Cambiar esta ruta si el modelo se encuentra en otro directorio.
RUTA_SUC = Path(os.environ.get(
    "SUC_PATH", str(RAIZ / "Lab9/embeddings-m-model.vec")
)).expanduser()
LIMITE_SUC = 200_000  # Usar 100_000 si hay poca memoria.

print(f"Raíz del proyecto: {RAIZ}")
print(f"Noticias: {RUTA_NOTICIAS.relative_to(RAIZ)}")
print(f"Quijote: {RUTA_QUIJOTE.relative_to(RAIZ)}")

Raíz del proyecto: /Users/josetanchezz/Desktop/NLP
Noticias: Lab1/df_total.csv
Quijote: Lab5/don-quijote.txt


## 1.2 Carga y revisión de los documentos

Cada noticia con texto es un documento; el libro completo es un documento. Se excluyen las noticias vacías y se reporta cuántas fueron descartadas. Se mantienen los textos repetidos para conservar el corpus usado en el Laboratorio 1, pero se informa su cantidad porque pueden aumentar la frecuencia de algunos contextos.

En el Quijote se retiran la cabecera y la licencia de Project Gutenberg. Se conservan el título, los preliminares y el texto literario en español.

In [2]:
df_original = pd.read_csv(RUTA_NOTICIAS)
if "news" not in df_original.columns:
    raise ValueError(f"No se encontró la columna news: {df_original.columns.tolist()}")

textos = df_original["news"].fillna("").astype(str).str.strip()
mascara_validos = textos.ne("")
noticias = df_original.loc[mascara_validos].copy()
noticias["news"] = textos.loc[mascara_validos]
noticias = noticias.reset_index(drop=True)
documentos_noticias = noticias["news"].tolist()

texto_quijote_original = RUTA_QUIJOTE.read_text(encoding="utf-8-sig")
inicio = texto_quijote_original.find("El ingenioso hidalgo don Quijote de la Mancha")
fin = re.search(
    r"(?:End of Project Gutenberg[^\n]*|\*\*\*\s*END OF (?:THIS|THE) PROJECT GUTENBERG)",
    texto_quijote_original,
)
if inicio == -1 or fin is None or fin.start() <= inicio:
    raise ValueError("Revisar los delimitadores de la edición del Quijote antes de continuar.")
texto_quijote = texto_quijote_original[inicio:fin.start()].strip()
documentos_quijote = [texto_quijote]

revision_carga = pd.DataFrame([
    {"Corpus": "Noticias", "Documentos originales": len(df_original),
     "Documentos utilizados": len(noticias),
     "Documentos vacíos excluidos": int((~mascara_validos).sum()),
     "Textos duplicados adicionales": int(noticias["news"].duplicated().sum())},
    {"Corpus": "Quijote", "Documentos originales": 1,
     "Documentos utilizados": 1, "Documentos vacíos excluidos": 0,
     "Textos duplicados adicionales": 0},
])
display(revision_carga)
print("Inicio del Quijote utilizado:\n", texto_quijote[:400])
print("\nFinal del Quijote utilizado:\n", texto_quijote[-300:])

,Corpus,Documentos originales,Documentos utilizados,Documentos vacíos excluidos,Textos duplicados adicionales
0,Noticias,1217,1213,4,76
1,Quijote,1,1,0,0


Inicio del Quijote utilizado:
 El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de
los que residen en su Consejo, certifico y doy fe que, habiendo visto por
los señores dél un libro intitulado El ingenioso hidalgo de la Mancha,
compuesto por Miguel de Cervantes Saavedra, tasaron cada pliego del dicho
libro a tres maravedís y medio; el cual tiene ochenta 

Final del Quijote utilizado:
 o de
sus escritos enteramente, como deseaba, pues no ha sido otro mi deseo que
poner en aborrecimiento de los hombres las fingidas y disparatadas
historias de los libros de caballerías, que, por las de mi verdadero don
Quijote, van ya tropezando, y han de caer del todo, sin duda alguna. Vale.



Fin


## 1.3 Decisiones de normalización

Retomamos la tokenización, las minúsculas y la eliminación de puntuación del Laboratorio 1, pero cambiamos las decisiones que afectarían el aprendizaje de contextos.

| Elemento | Decisión | Justificación |
|---|---|---|
| Oraciones | Segmentar antes de eliminar puntuación | Evita crear pares entre el final de una oración y el inicio de otra. |
| Mayúsculas | Convertir a minúsculas | Agrupa variantes y reduce la dispersión en corpus pequeños. Se comprobará su cobertura en SUC. |
| Puntuación | Retirar después de segmentar | Las ventanas se construirán sobre palabras y números. |
| Números | Conservar secuencias de dígitos, sin un token artificial | Mantiene referencias numéricas y evita introducir un marcador sin vector en SUC. Los separadores decimales se eliminan y separan los grupos numéricos. |
| Tildes, diéresis y ñ | Conservar y normalizar Unicode a NFC | Evita confundir palabras como `sí` y `si` o `año` y `ano`. |
| Stopwords | Conservar | Quitarlas acerca palabras que no eran vecinas y elimina información sintáctica. El submuestreo de Word2Vec podrá reducir su peso durante el entrenamiento. |
| Lematización | No aplicar | Conserva número, género y tiempos verbales, necesarios para estudiar relaciones morfológicas y analogías. |

`gobiernos` y `gobernó` permanecen distintos: el primero es un sustantivo plural y el segundo, un verbo en pasado. Una lematización correcta daría `gobierno` y `gobernar`, respectivamente; agruparlos en una sola forma borraría también la diferencia de categoría gramatical.

**Compatibilidad con SUC.** Las consultas usarán exactamente estos tokens, sin retirar tildes ni aplicar transformaciones adicionales a un solo modelo. La comprobación de la sección 1.7 mostrará qué grafías están realmente presentes en el vocabulario cargado; no se presupone el resultado.

In [3]:
# Punkt segmenta en español sin necesitar un modelo de spaCy.
try:
    nltk.sent_tokenize("Primera oración. Segunda oración.", language="spanish")
except LookupError:
    nltk.download("punkt", quiet=True, raise_on_error=True)
    nltk.download("punkt_tab", quiet=True, raise_on_error=True)

PATRON_TOKEN = re.compile(r"[a-záéíóúüñ0-9]+", flags=re.IGNORECASE)


def preparar_documento(texto):
    """Segmentar primero y normalizar después, conservando el orden."""
    texto = unicodedata.normalize("NFC", texto)
    # En el CSV hay finales concatenados, como «imaginamos.Para».
    # Esta reparación es heurística; puede afectar abreviaturas sin espacio.
    texto = re.sub(r"(?<=[.!?])(?=[A-ZÁÉÍÓÚÜÑ¿¡])", " ", texto)
    resultado = []
    for oracion in nltk.sent_tokenize(texto, language="spanish"):
        tokens = PATRON_TOKEN.findall(oracion.lower())
        if tokens:
            resultado.append(tokens)
    return resultado


# Se conserva la estructura por documento para usos posteriores.
noticias["oraciones_tokens"] = [
    preparar_documento(texto) for texto in documentos_noticias
]
quijote_por_documento = [preparar_documento(texto) for texto in documentos_quijote]

# Estas dos listas de listas son las entradas para gensim.
oraciones_noticias = [
    oracion for documento in noticias["oraciones_tokens"] for oracion in documento
]
oraciones_quijote = [
    oracion for documento in quijote_por_documento for oracion in documento
]
corpus_oraciones = {"Noticias": oraciones_noticias, "Quijote": oraciones_quijote}

print("Formato de entrada: list[list[str]]")
for nombre, oraciones in corpus_oraciones.items():
    print(f"\n{nombre}: primeras dos oraciones tokenizadas")
    for oracion in oraciones[:2]:
        print(oracion)

Formato de entrada: list[list[str]]

Noticias: primeras dos oraciones tokenizadas
['durante', 'el', 'foro', 'la', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el', 'director', 'de', 'sostenibilidad', 'y', 'clientes', 'globales', 'de', 'bbva', 'en', 'colombia', 'andrés', 'garcía', 'aseguró', 'que', 'es', 'importante', 'entender', 'que', 'la', 'sostenibilidad', 'no', 'la', 'podemos', 'asociar', 'a', 'mayores', 'costos']
['yo', 'creo', 'que', 'el', 'no', 'tener', 'un', 'concepto', 'de', 'negocio', 'sostenible', 'puede', 'tener', 'un', 'mayor', 'impacto', 'de', 'lo', 'que', 'imaginamos']

Quijote: primeras dos oraciones tokenizadas
['el', 'ingenioso', 'hidalgo', 'don', 'quijote', 'de', 'la', 'mancha', 'tasa', 'yo', 'juan', 'gallo', 'de', 'andrada', 'escribano', 'de', 'cámara', 'del', 'rey', 'nuestro', 'señor', 'de', 'los', 'que', 'residen', 'en', 'su', 'consejo', 'certifico', 'y', 'doy', 'fe', 'que', 'habiendo', 'visto', 'por', 'los', 'señores', 'dél', 

La segmentación de oraciones es automática y puede equivocarse con abreviaturas, encabezados o puntuación irregular. En el libro los saltos de línea de la edición no se toman por sí solos como finales de oración.

## 1.4 Tamaño de los corpus

Un **token** es una aparición de una palabra o número; un **tipo** es una forma distinta después de normalizar. Se cuentan solamente las oraciones que conservan al menos un token.

In [4]:
frecuencias_corpus = {
    nombre: Counter(token for oracion in oraciones for token in oracion)
    for nombre, oraciones in corpus_oraciones.items()
}
documentos_por_corpus = {"Noticias": len(documentos_noticias), "Quijote": 1}
resumen_corpus = pd.DataFrame([
    {"Corpus": nombre, "Documentos": documentos_por_corpus[nombre],
     "Oraciones": len(oraciones),
     "Tokens": sum(frecuencias_corpus[nombre].values()),
     "Tipos": len(frecuencias_corpus[nombre])}
    for nombre, oraciones in corpus_oraciones.items()
])
display(resumen_corpus)

,Corpus,Documentos,Oraciones,Tokens,Tipos
0,Noticias,1213,21855,637610,29507
1,Quijote,1,9550,381230,22943


## 1.5 Vocabulario y frecuencia mínima

Se construye un vocabulario independiente para cada corpus y para cada valor de `min_count`. El porcentaje excluido se calcula sobre **apariciones de tokens**, no sobre tipos:

$$\%\text{ de tokens fuera}=100\frac{\sum_{w:f(w)<m} f(w)}{\sum_w f(w)}.$$

En esta etapa se mide la cobertura; las listas originales no se filtran ni se reordenan. Gensim aplicará el umbral al construir su vocabulario durante el entrenamiento.

In [5]:
MIN_COUNTS = (1, 2, 5)
vocabularios = {}
filas_vocabulario = []
for nombre, frecuencias in frecuencias_corpus.items():
    total = sum(frecuencias.values())
    vocabularios[nombre] = {}
    for minimo in MIN_COUNTS:
        vocabulario = {palabra for palabra, frecuencia in frecuencias.items()
                       if frecuencia >= minimo}
        vocabularios[nombre][minimo] = vocabulario
        fuera = sum(frecuencia for palabra, frecuencia in frecuencias.items()
                    if palabra not in vocabulario)
        filas_vocabulario.append({
            "Corpus": nombre, "min_count": minimo,
            "Tamaño del vocabulario": len(vocabulario),
            "Tokens fuera": fuera,
            "Tokens fuera (%)": round(100 * fuera / total, 4) if total else 0.0,
        })
resumen_vocabularios = pd.DataFrame(filas_vocabulario)
display(resumen_vocabularios)

# Ejemplo conceptual tokens -> IDs. Gensim asignará sus propios IDs.
palabra_a_id = {palabra: i for i, (palabra, frecuencia) in enumerate(
    frecuencias_corpus["Noticias"].most_common()
)}
ejemplo_tokens = oraciones_noticias[0]
display(pd.DataFrame({"Token": ejemplo_tokens,
                      "ID ilustrativo": [palabra_a_id[t] for t in ejemplo_tokens]}))

,Corpus,min_count,Tamaño del vocabulario,Tokens fuera,Tokens fuera (%)
0,Noticias,1,29507,0,0.0000
1,Noticias,2,17129,12378,1.9413
2,Noticias,5,8890,34319,5.3824
3,Quijote,1,22943,0,0.0000
4,Quijote,2,11760,11183,2.9334
5,Quijote,5,5301,28121,7.3764


,Token,ID ilustrativo
0,durante,61
1,el,3
2,foro,1582
3,la,1
4,banca,206
5,articulador,12811
6,empresarial,667
7,para,11
8,el,3
9,desarrollo,80


## 1.6 Pares skip-gram con código propio

Cada posición central se combina con todas las posiciones de contexto a una distancia máxima de dos tokens. Los pares son **dirigidos** y se mantienen sus repeticiones: dos posiciones distintas pueden producir el mismo par de palabras. Nunca se atraviesa el límite de una oración.

Se elige una oración de entre 8 y 15 tokens para poder mostrar **todos** sus pares sin truncar la tabla.

In [6]:
def pares_skipgram(tokens, ventana=2):
    """Producir pares (central, contexto), sin incluir la propia posición."""
    if ventana < 1:
        raise ValueError("La ventana debe ser positiva.")
    for i, central in enumerate(tokens):
        inicio = max(0, i - ventana)
        fin = min(len(tokens), i + ventana + 1)
        for j in range(inicio, fin):
            if i != j:
                yield central, tokens[j]


oracion_ejemplo = next(
    (oracion for oracion in oraciones_noticias if 8 <= len(oracion) <= 15),
    oraciones_noticias[0],
)
pares_ejemplo = list(pares_skipgram(oracion_ejemplo, ventana=2))
tabla_pares = pd.DataFrame(pares_ejemplo, columns=["Palabra central", "Contexto"])
tabla_pares.index = range(1, len(tabla_pares) + 1)
tabla_pares.index.name = "Par"
print("Oración normalizada:", " ".join(oracion_ejemplo))
print(f"Tokens: {len(oracion_ejemplo)} | Pares con ventana 2: {len(pares_ejemplo)}")
with pd.option_context("display.max_rows", None):
    display(tabla_pares)

Oración normalizada: la sostenibilidad nos abre oportunidades a mejores fuentes de financiamiento agregó
Tokens: 11 | Pares con ventana 2: 38


,Palabra central,Contexto
Par,,
1,la,sostenibilidad
2,la,nos
3,sostenibilidad,la
4,sostenibilidad,nos
5,sostenibilidad,abre
6,nos,la
7,nos,sostenibilidad
8,nos,abre
9,nos,oportunidades


### Cantidad de pares en todo el corpus de noticias

Para una oración de longitud $n$, cada distancia $d$ aporta $2(n-d)$ pares dirigidos. Con $m=\min(w,n-1)$:

$$N(n,w)=2\sum_{d=1}^{m}(n-d)=2mn-m(m+1).$$

Esta fórmula permite contar sin guardar millones de pares en memoria. Es el conteo de una pasada con **ventana fija**, sobre todos los tokens preparados, antes de `min_count` o submuestreo. El entrenamiento de gensim puede usar ventanas reducidas aleatoriamente, descartar tokens y repetir épocas, por lo que su número efectivo de ejemplos puede ser diferente.

In [7]:
def contar_pares(longitud, ventana):
    m = min(ventana, max(0, longitud - 1))
    return 2 * m * longitud - m * (m + 1)


# Comprobar la fórmula contra el generador usado en el ejemplo.
assert contar_pares(len(oracion_ejemplo), 2) == len(pares_ejemplo)
total_tokens_noticias = sum(map(len, oraciones_noticias))
comparacion_pares = pd.DataFrame([
    {"Ventana": ventana, "Tokens": total_tokens_noticias,
     "Pares de entrenamiento": sum(contar_pares(len(s), ventana)
                                    for s in oraciones_noticias)}
    for ventana in (2, 5)
])
comparacion_pares["Pares por token"] = (
    comparacion_pares["Pares de entrenamiento"] / total_tokens_noticias
).round(4)
display(comparacion_pares)

,Ventana,Tokens,Pares de entrenamiento,Pares por token
0,2,637610,2419436,3.7945
1,5,637610,5722750,8.9753


**¿Por qué Word2Vec no necesita datos etiquetados?** El propio orden del texto proporciona la señal de entrenamiento: cada palabra central permite obtener sus contextos observados. No se necesitan categorías anotadas por una persona. Con una ventana mayor cada posición puede originar más pares; cerca de los límites de oración hay menos vecinos disponibles. Este aprendizaje se denomina autosupervisado.

## 1.7 Comprobación del vocabulario fastText SUC

Se carga SUC una sola vez por sesión, con un límite de vocabulario para controlar la memoria. Si el archivo todavía no está disponible, esta celda deja explícita la comprobación pendiente y permite ejecutar el resto de la sección.

Se consulta `key_to_index` para comprobar la presencia **exacta** de cada grafía. La ausencia de una palabra en las primeras 200 000 entradas no demuestra que esté ausente del archivo completo. La cobertura se mide sin alterar los tokens de los corpus.

In [8]:
if not RUTA_SUC.is_file():
    print("PENDIENTE: no se encontró el archivo de fastText SUC.")
    print(f"Colócalo en {RUTA_SUC} o modifica RUTA_SUC y vuelve a ejecutar esta celda.")
else:
    from gensim.models import KeyedVectors

    firma_suc = (str(RUTA_SUC.resolve()), LIMITE_SUC)
    if globals().get("_firma_suc_cargada") != firma_suc:
        suc = KeyedVectors.load_word2vec_format(
            str(RUTA_SUC), limit=LIMITE_SUC
        )
        _firma_suc_cargada = firma_suc

    variantes = ["españa", "España", "espana", "ESPAÑA",
                 "tecnología", "tecnologia", "año", "ano"]
    revision_suc = pd.DataFrame([
        {"Grafía exacta": palabra, "Presente en SUC cargado": palabra in suc.key_to_index}
        for palabra in variantes
    ])
    print(f"Entradas cargadas: {len(suc):,} | Dimensiones: {suc.vector_size}")
    display(revision_suc)

    cobertura_suc = pd.DataFrame([
        {"Corpus": nombre,
         "Tipos cubiertos (%)": round(100 * sum(t in suc.key_to_index for t in frec)
                                      / len(frec), 2),
         "Tokens cubiertos (%)": round(100 * sum(n for t, n in frec.items()
                                                 if t in suc.key_to_index)
                                       / sum(frec.values()), 2)}
        for nombre, frec in frecuencias_corpus.items()
    ])
    display(cobertura_suc)
    print("Interpretar estas tablas antes de cerrar la justificación de compatibilidad con SUC.")

PENDIENTE: no se encontró el archivo de fastText SUC.
Colócalo en /Users/josetanchezz/Desktop/NLP/Lab9/embeddings-m-model.vec o modifica RUTA_SUC y vuelve a ejecutar esta celda.


## 1.8 Análisis de los resultados

*Pendiente de redactar después de ejecutar y revisar las salidas.*

El análisis recogerá el tamaño de cada corpus, el efecto de `min_count`, la comparación entre ventanas y la evidencia de compatibilidad con SUC. Las decisiones metodológicas anteriores ya están documentadas; las conclusiones numéricas se completarán con los resultados reales.